## Sección 1. Listas

Una lista es una colección **ordenada y modificable**. En ingeniería de datos aparece en todas partes: los ficheros pendientes de un lote, las filas leídas de un CSV, las métricas de una ejecución o la cola de tareas de un orquestador.

### Ejemplos resueltos

#### Ejemplo 1.1 · Gestionar los ficheros de un lote de ingesta

Cada noche llegan a la zona de aterrizaje los ficheros de ventas del día. El proceso de ingesta mantiene una lista con los ficheros pendientes.

In [0]:
ficheros = ["ventas_0101.csv","ventas_0102.csv","ventas_0104.csv"]
# añadir un fichero al final 
ficheros.append("ventas_0105.csv")

#insertar el fichero del dia 3
ficheros.insert(2,"ventas_0103.csv")

print (ficheros)

print("Fichero en el lote",len(ficheros))
print("Primero:",ficheros[0], "| Ultimo:",ficheros[-1])
print("Dos ultimos",ficheros[-2:])

#eliminamos el primer fichero
ficheros.remove("ventas_0101.csv")
print(ficheros)
print("Posicion de ventas_0104.csv",ficheros.index("ventas_0104.csv"))


['ventas_0101.csv', 'ventas_0102.csv', 'ventas_0103.csv', 'ventas_0104.csv', 'ventas_0105.csv']
Fichero en el lote 5
Primero: ventas_0101.csv | Ultimo: ventas_0105.csv
Dos ultimos ['ventas_0104.csv', 'ventas_0105.csv']
['ventas_0102.csv', 'ventas_0103.csv', 'ventas_0104.csv', 'ventas_0105.csv']
Posicion de ventas_0104.csv 2


**Paso a paso**

1. `append()` añade siempre al final de la lista.
2. `insert(2, ...)` coloca el elemento en la posición 2 y desplaza el resto a la derecha. Así el fichero que llegó tarde queda en su sitio.
3. `len()` devuelve el número de elementos. Los índices empiezan en 0, y los negativos cuentan desde el final: `[-1]` es el último.
4. El *slicing* `[-2:]` devuelve una **lista nueva** con los dos últimos elementos.
5. `remove()` elimina la primera aparición del valor indicado. Si el valor no existe, lanza `ValueError`.
6. `index()` devuelve la posición del valor. Como ya hemos quitado el primer fichero, `ventas_0104.csv` pasa a la posición 2.

#### Ejemplo 1.2 · Estadísticas de latencia de una API de origen

Un pipeline lee datos de una API y guarda cuánto tarda cada petición. Queremos un pequeño informe de rendimiento.

In [0]:
latencias_ms = [120, 85, 430, 95, 110, 1500, 88, 102, 97, 640]

total =sum(latencias_ms)
media = total/len(latencias_ms)

print("Nº de latencias:", len(latencias_ms))
print("Minimo", min(latencias_ms),"| Maximo", max(latencias_ms))
print("Media", round(media,1),"ms")

ordenadas = sorted(latencias_ms)
n=len(ordenadas)
mediana = (ordenadas[n //2-1] + ordenadas[n//2])/2

print("Mediana", round(mediana,1),"ms")

print("Tres mas lentas:",sorted(latencias_ms,reverse=True)[:3])
print("Original sin tocar", latencias_ms)

Nº de latencias: 10
Minimo 85 | Maximo 1500
Media 326.7 ms
Mediana 106.0 ms
Tres mas lentas: [1500, 640, 430]
Original sin tocar [120, 85, 430, 95, 110, 1500, 88, 102, 97, 640]


**Paso a paso**

1. `sum()`, `min()`, `max()` y `len()` son funciones integradas que aceptan una lista.
2. La media (326,7 ms) se dispara por dos peticiones muy lentas (1500 y 640 ms). Por eso en monitorización se mira también la **mediana**, que no se deja arrastrar por los valores extremos.
3. Para la mediana hace falta ordenar. `sorted()` devuelve una lista nueva y deja la original intacta. Con 10 valores, la mediana es la media de los dos centrales: posiciones 4 y 5 de la lista ordenada (102 y 110).
4. `sorted(..., reverse=True)[:3]` ordena de mayor a menor y se queda con los tres primeros.
5. La última línea confirma que `latencias_ms` conserva su orden original. Si hubiéramos usado `latencias_ms.sort()`, la lista original sí habría cambiado.

#### Ejemplo 1.3 · Alias y copia: no estropear la capa bronce

En una arquitectura medallón, la capa **bronce** guarda los datos tal como llegan y la capa **plata** guarda los datos limpios. La regla es no modificar nunca bronce.

In [0]:
capa_bronce = [100, -5, 230, 0, 75]

#error comun parece ser una copia pero es la misma lista

capa_plata = capa_bronce
capa_plata.remove(-5)
print("capa_bronce",capa_bronce)
print("capa_plata",capa_plata)
print("misma lista", capa_plata is capa_bronce)

#metodo correcto de copiar 
capa_bronce=[100, -5,230,0,75]
capa_plata = capa_bronce.copy()
capa_plata.remove(-5)
print("capa_bronce",capa_bronce)
print("capa_plata",capa_plata)
print("misma lista", capa_plata is capa_bronce)

capa_bronce [100, 230, 0, 75]
capa_plata [100, 230, 0, 75]
misma lista True
capa_bronce [100, -5, 230, 0, 75]
capa_plata [100, 230, 0, 75]
misma lista False


**Paso a paso**

1. `capa_plata = capa_bronce` **no copia** la lista. Crea un segundo nombre (un *alias*) para el mismo objeto en memoria.
2. Por eso, al quitar el `5` desde `capa_plata`, también desaparece de `capa_bronce`. El operador `is` lo confirma: devuelve `True` porque ambas variables apuntan al mismo objeto.
3. Con `.copy()` (o `[:]`) se crea una lista nueva con el mismo contenido. Ahora los cambios en plata no afectan a bronce.

> Este error es muy frecuente en los pipelines: una función de limpieza recibe una lista y, sin querer, modifica los datos originales.
>

#### Ejemplo 1.4 · Cola de jobs (FIFO) y pila de migraciones (LIFO)

Un orquestador ejecuta los jobs en el orden en que llegan, como una **cola**. En cambio, cuando hay que deshacer migraciones de base de datos, se revierte primero la última aplicada, como en una **pila**.

In [0]:
from collections import deque
#cola fifo
cola_jobs = deque()
cola_jobs.append("ingesta_clientes")
cola_jobs.append("ingesta_pedidos")
cola_jobs.append("calculo_kpis")
print("cola",list(cola_jobs))
siguiente = cola_jobs.popleft()
print("El siguiente en ejecutarse es:",siguiente)
print("Pendientes:",list(cola_jobs))
#pila lifo
migraciones=[]
migraciones.append("v1_crear_tablas")
migraciones.append("v2_add_columna_email")
migraciones.append("v3_crear_indice")
print("Aplicadas:", migraciones)
while migraciones:
    print("Revertiendo",migraciones.pop())
print("Quedan",migraciones)


cola ['ingesta_clientes', 'ingesta_pedidos', 'calculo_kpis']
El siguiente en ejecutarse es: ingesta_clientes
Pendientes: ['ingesta_pedidos', 'calculo_kpis']
Aplicadas: ['v1_crear_tablas', 'v2_add_columna_email', 'v3_crear_indice']
Revertiendo v3_crear_indice
Revertiendo v2_add_columna_email
Revertiendo v1_crear_tablas
Quedan []


**Paso a paso**

1. Quitar el primer elemento de una lista con `pop(0)` es lento en listas grandes, porque Python tiene que desplazar todos los demás. Para colas se usa `deque` del módulo `collections`, cuyo `popleft()` es inmediato.
2. Para la pila basta una lista normal: `append()` apila y `pop()` sin argumento desapila el último elemento.
3. `while migraciones:` se repite mientras la lista tenga elementos, porque una lista vacía se evalúa como `False`.

#### Ejemplo 1.5 · Lista de listas: filas de un CSV

Cuando se lee un CSV sin librerías, cada fila es una lista y el fichero completo es una lista de listas.


In [0]:
from operator import  itemgetter

filas = [
    ["2026-10-01", "MAD01", 1250.0],
    ["2026-10-01", "BCN02", 980.5],
    ["2026-10-02", "MAD01", 1530.0],
    ["2026-10-02", "VLC03", 410.0],
    ["2026-10-03", "BCN02", 1105.0],
]
print("primera fila",filas[0])
print("tienda de la segunda fila:", filas[1][1])

importes = []
for fila in filas:
    importes.append(fila[2])
print("Columna importe:", importes)
print("Total vendido ", sum(importes))
por_importe = sorted(filas, key=itemgetter(2),reverse=True)
print("Venta mas alta ",por_importe[0])
print("venta mas baja", por_importe[-1])

primera fila ['2026-10-01', 'MAD01', 1250.0]
tienda de la segunda fila: BCN02
Columna importe: [1250.0, 980.5, 1530.0, 410.0, 1105.0]
Total vendido  5275.5
Venta mas alta  ['2026-10-02', 'MAD01', 1530.0]
venta mas baja ['2026-10-02', 'VLC03', 410.0]


**Paso a paso**

1. `filas[1][1]` accede primero a la fila 1 y, dentro de ella, a la columna 1 (la tienda).
2. Para extraer una columna recorremos las filas y guardamos el elemento 2 de cada una.
3. `itemgetter(2)` (del módulo `operator`) crea una función que, dada una fila, devuelve su elemento 2. `sorted()` la usa como **criterio de ordenación** mediante el parámetro `key`.

#### Ejemplo 1.6 · Trocear un fichero en bloques de HDFS

HDFS no guarda los ficheros enteros: los divide en **bloques** de 128 MB y guarda **3 copias** (réplicas) de cada bloque en nodos distintos (DataNodes). Así, si un nodo falla, los datos siguen disponibles en otro.

In [0]:
tamano_fichero_mb = 700
tamano_bloque_mb = 128
replicas = 3
nodos = ["datanode1", "datanode2", "datanode3", "datanode4"]
#particionado del fichero por el tamaño
bloques = []
restante = tamano_fichero_mb
while restante > 0:
    bloques.append(min(tamano_bloque_mb, restante))
    restante -= tamano_bloque_mb
print("Bloques (MB):", bloques)
print("Número de bloques:", len(bloques))
#guardado de el contenido
contenido =[]
for _ in nodos:
    contenido.append([])

#repartir los bloques por las replicas 
for j in range(len(bloques)):
    for y in range(replicas):
        destino =(j+y)% len(nodos)
        contenido[destino].append(f"blk{j}")
for i in range(len(nodos)):
    print(nodos[i],contenido[i])
print("Espacio ocupado en el cluster:",sum(bloques)*replicas,"MB")

Bloques (MB): [128, 128, 128, 128, 128, 60]
Número de bloques: 6
datanode1 ['blk0', 'blk2', 'blk3', 'blk4']
datanode2 ['blk0', 'blk1', 'blk3', 'blk4', 'blk5']
datanode3 ['blk0', 'blk1', 'blk2', 'blk4', 'blk5']
datanode4 ['blk1', 'blk2', 'blk3', 'blk5']
Espacio ocupado en el cluster: 2100 MB


**Paso a paso**

1. El bucle `while` va restando 128 MB hasta agotar el fichero. `min()` hace que el último bloque tenga solo lo que queda (60 MB). Los bloques no se rellenan hasta 128 MB.
2. `contenido` es una lista de listas: una lista por cada DataNode.
3. La réplica `r` del bloque `b` va al nodo `(b + r) % 4`. El operador `%` hace que, al llegar al último nodo, se vuelva a empezar por el primero, y garantiza que las 3 réplicas de un bloque caen en nodos distintos.
4. Un fichero de 700 MB ocupa 2100 MB en el clúster. Es el precio de la tolerancia a fallos.

#### Ejemplo 1.7 · Detectar *data skew* entre particiones

En Spark, cada partición la procesa una tarea, y una etapa no termina hasta que acaba **la tarea más lenta**. Si unas particiones tienen muchos más datos que otras (*data skew*), todo el job espera por ellas.

In [0]:
registros_por_particion = [10250, 9870, 10110, 48300, 9930, 10020, 9990, 51200]

total = sum(registros_por_particion)
media = total / len(registros_por_particion)
print("Particiones:", len(registros_por_particion), "| Registros:", total)
print("Media por partición:", round(media))

mayor = max(registros_por_particion)
print("Partición más grande:", registros_por_particion.index(mayor), "con", mayor, "registros")
print("Factor de skew (máx / media):", round(mayor / media, 2))

sesgadas = []
for i in range(len(registros_por_particion)):
    if registros_por_particion[i] > 2 * media:
        sesgadas.append(i)
print("Particiones con más del doble de la media:", sesgadas)

dos_mayores = sorted(registros_por_particion, reverse=True)[:2]
print("Las 2 mayores concentran el", round(sum(dos_mayores) / total * 100, 1), "% de los datos")

Particiones: 8 | Registros: 159670
Media por partición: 19959
Partición más grande: 7 con 51200 registros
Factor de skew (máx / media): 2.57
Particiones con más del doble de la media: [3, 7]
Las 2 mayores concentran el 62.3 % de los datos


**Paso a paso**

1. Con la suma y la longitud de la lista se obtiene la media de registros por partición.
2. El **factor de skew** compara la partición más grande con la media. Un valor cercano a 1 indica un reparto equilibrado; aquí es 2,57.
3. El bucle con `range(len(...))` recorre los índices para poder informar del **número de partición**, no solo de su tamaño.
4. Dos particiones de ocho concentran más de la mitad de los datos: esas dos tareas marcarán la duración del job. La solución habitual es reparticionar (Ejercicio 1.6).

#### Ejercicio 1.1 · Lecturas de un sensor fuera de rango

Un sensor de temperatura envía `-999.0` cuando falla, y a veces envía valores imposibles. El rango físico válido es de -40 a 85 ºC, ambos incluidos.

In [0]:
lecturas = [21.5, 22.1, -999.0, 23.4, 150.2, 22.8, -999.0, 24.0, 21.9, 87.3]

validas = []
descartadas = []

for l in lecturas:
    if -40 <= l <= 85:
        validas.append(l)
    else:
        descartadas.append(l)

porc_descart = (len(descartadas) / len(lecturas)) * 100
media_val = sum(validas) / len(validas)
codigos_error = lecturas.count(-999.0)

print(f"Válidas: {validas}")
print(f"Descartadas: {descartadas}")
print(f"Porcentaje descartado: {porc_descart:.1f} %")
print(f"Media de las válidas: {media_val:.2f}")
print(f"Códigos -999.0 (sensor caído): {codigos_error}")

Válidas: [21.5, 22.1, 23.4, 22.8, 24.0, 21.9]
Descartadas: [-999.0, 150.2, -999.0, 87.3]
Porcentaje descartado: 40.0 %
Media de las válidas: 22.62
Códigos -999.0 (sensor caído): 2


#### Ejercicio 1.2 · Quitar duplicados conservando el orden de llegada

Una API de clientes devuelve identificadores repetidos. Hay que eliminarlos **conservando el orden en que aparecen por primera vez**. Resuélvelo solo con listas (los `set` se ven en la Sección 3).


In [0]:
ids_recibidos = ["C003", "C001", "C003", "C007", "C001", "C002", "C007", "C003"]
unicos =[]

for id in ids_recibidos:
    if id not in unicos:
        unicos.append(id)

print(f"IDs únicos: {unicos}")
print(f"Número de IDs únicos: {len(unicos)}")

IDs únicos: ['C003', 'C001', 'C007', 'C002']
Número de IDs únicos: 4


#### Ejercicio 1.3 · Dividir registros en lotes para una API

La API de destino acepta como máximo 5 registros por petición. Divide la lista en lotes con *slicing* y muestra cada lote con su número y su tamaño. El último lote puede ser más pequeño. Calcula el número de lotes con una fórmula, sin contar a mano.

In [0]:
registros = list(range(1, 24))  # 23 registros: 1, 2, ..., 23
tamano_lote = 5

total_registros = len(registros)
num_lotes = (total_registros + tamano_lote - 1) // tamano_lote

for i in range(num_lotes):
    inicio = i * tamano_lote
    fin = inicio + tamano_lote
    lote = registros[inicio:fin]
    print(f"Lote {i + 1}: {len(lote)} registros -> {lote}")

print("Número de lotes:", num_lotes)

Lote 1: 5 registros -> [1, 2, 3, 4, 5]
Lote 2: 5 registros -> [6, 7, 8, 9, 10]
Lote 3: 5 registros -> [11, 12, 13, 14, 15]
Lote 4: 5 registros -> [16, 17, 18, 19, 20]
Lote 5: 3 registros -> [21, 22, 23]
Número de lotes: 5


#### Ejercicio 1.4 · El bug de borrar mientras se recorre
Explica por qué siguen apareciendo 35 y 60. Pista: piensa qué le ocurre al índice interno del for cuando se elimina un elemento.

al eliminar el indice 1 elimina -10 tras esa accion todos los elementos se desplazan entonces el -35 ocupa la posicion 1 al volver a ejecutarse el bulcle se salta el indice 1 y pasa al 2 que ahora esta el 80 en lugar del -35 por eso se salta valores 

In [0]:
importes = [120, -10, -35, 80, -5, -60, 200]

limpios = []
for x in importes:
    if x >= 0:
        limpios.append(x)

importes = limpios
print(importes)

[120, 80, 200]


Ejercicio 1.5 · Consultas sobre una lista de pedidos
1. Obtener la lista de importes de los pedidos `ENTREGADO` de la tienda `MAD01` y su total.
2. Obtener los identificadores de los 3 pedidos de mayor importe, sea cual sea su estado.
3. Cambiar **en la propia lista** el estado `PENDIENTE` por `EN_CURSO` y mostrar la columna de estados.

In [0]:
from operator import itemgetter
#  [pedido_id, tienda, importe, estado]
pedidos = [
    ["P01", "MAD01", 250.0, "ENTREGADO"],
    ["P02", "BCN02", 89.9, "PENDIENTE"],
    ["P03", "MAD01", 1200.0, "ENTREGADO"],
    ["P04", "VLC03", 45.5, "CANCELADO"],
    ["P05", "MAD01", 310.0, "PENDIENTE"],
    ["P06", "BCN02", 760.0, "ENTREGADO"],
    ["P07", "MAD01", 99.0, "ENTREGADO"],
]
importes_mad01=[]
for ped in pedidos:
    if ped[1] == "MAD01" and ped[3] == "ENTREGADO":
        importes_mad01.append(ped[2])
print(f"Importes entregados en MAD01: {importes_mad01} -> total {sum(importes_mad01)}")

pedidos_ord= sorted(pedidos, key= itemgetter(2), reverse =True)
top_3 = []
for ped in pedidos_ord[:3]:
    top_3.append(ped[0])
print(f"Top 3 pedidos por importe: {top_3}")

estados_actualizados = []
for p in pedidos:
    if p[3] == "PENDIENTE":
        p[3] = "EN_CURSO"
    estados_actualizados.append(p[3])

print("Estados actualizados:", estados_actualizados)

Importes entregados en MAD01: [250.0, 1200.0, 99.0] -> total 1549.0
Top 3 pedidos por importe: ['P03', 'P06', 'P05']
Estados actualizados: ['ENTREGADO', 'EN_CURSO', 'ENTREGADO', 'CANCELADO', 'EN_CURSO', 'ENTREGADO', 'ENTREGADO']


#### Ejercicio 1.6 · Reparticionar con *round-robin*

La función `repartition()` de Spark redistribuye los registros para equilibrar las particiones. Vamos a simular la versión más sencilla, el reparto *round-robin*.
1. Junta todos los registros en una sola lista, en orden. Puedes usar `extend()`.
2. Crea 4 particiones vacías y asigna el registro número `i` a la partición `i % num_particiones`.
3. Muestra el contenido de cada partición y compara el factor de skew (máximo / media) antes y después, con dos decimales.

In [0]:
entrada = [
    [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    [113, 114],
    [115, 116, 117, 118, 119, 120],
]
num_particiones = 4

tamanos_antes = [len(p) for p in entrada]
media_antes = sum(tamanos_antes) / len(tamanos_antes)
skew_antes = max(tamanos_antes) / media_antes

registros = []
for particion in entrada:
    registros.extend(particion)

salida = [[] for _ in range(num_particiones)]
for i, reg in enumerate(registros):
    destino = i % num_particiones
    salida[destino].append(reg)

for i, particion in enumerate(salida):
    print(f"Partición {i}: {particion}")

tamanos_despues = [len(p) for p in salida]
media_despues = sum(tamanos_despues) / len(tamanos_despues)
skew_despues = max(tamanos_despues) / media_despues

print(f"Tamaños antes: {tamanos_antes} -> skew {skew_antes:.1f}")
print(f"Tamaños después: {tamanos_despues} -> skew {skew_despues:.1f}")

Partición 0: [101, 105, 109, 113, 117]
Partición 1: [102, 106, 110, 114, 118]
Partición 2: [103, 107, 111, 115, 119]
Partición 3: [104, 108, 112, 116, 120]
Tamaños antes: [12, 2, 6] -> skew 1.8
Tamaños después: [5, 5, 5, 5] -> skew 1.0


#### Ejercicio 1.7 · Ventanas deslizantes sobre un stream de eventos

Un sistema de streaming cuenta los eventos que llegan cada minuto. Para suavizar el ruido se calcula una **media móvil** con una ventana de 3 minutos.
1. Calcula con *slicing* la media de cada ventana de 3 minutos consecutivos (minutos 0-2, 1-3, 2-4…), redondeada a un decimal.
2. Indica qué ventana tiene la media más alta.
3. Detecta los minutos con **pico**: aquellos cuyo valor supera el doble de la media de los 3 minutos anteriores. Empieza a comprobar en el minuto 3.

In [0]:
eventos_por_minuto = [120, 135, 128, 410, 140, 132, 125, 390, 395, 130]
ventana = 3

medias_moviles = []

limite = len(eventos_por_minuto) -ventana 
for i in range(limite+1):
    trozo = eventos_por_minuto[i:i+ventana]
    media = round(sum(trozo)/ventana,1)
    medias_moviles.append(media)

print(f"Medias moviles {medias_moviles}")
print(f"Nº de ventanas calculadas {len(medias_moviles)}")


max_media = max(medias_moviles)
indice_media = medias_moviles.index(max_media)
print(f"Ventana más cargada: minutos {indice_media}-{indice_media + ventana - 1} (media {max_media})")

picos = []
for i in range(ventana, len(eventos_por_minuto)):
    media_previa = sum(eventos_por_minuto[i - ventana:i]) / ventana
    if eventos_por_minuto[i] > 2 * media_previa:
        picos.append(i)

print("Minutos con pico:", picos)


Medias moviles [127.7, 224.3, 226.0, 227.3, 132.3, 215.7, 303.3, 305.0]
Nº de ventanas calculadas 8
Ventana más cargada: minutos 7-9 (media 305.0)
Minutos con pico: [3, 7]


## Sección 2. Tuplas

Una tupla es una colección **ordenada e inmutable**: una vez creada, no se puede cambiar. Por eso encaja muy bien para representar un registro cerrado, una clave compuesta o una fila que ningún paso del pipeline debe alterar por accidente.

### Ejemplos resueltos

#### Ejemplo 2.1 · Un evento de sensor como registro inmutable


In [0]:
evento = ("2026-10-01T08:15:00", "sensor_07", 21.4)

print("Evento completo:", evento)
print("Sensor:", evento[1])
print("Campos:", len(evento))

# Desempaquetado: cada campo en su variable
momento, sensor, temperatura = evento
print(sensor, "midió", temperatura, "ºC a las", momento[11:16])

# Una tupla no se puede modificar:
# evento[2] = 22.0   ->  TypeError: 'tuple' object does not support item assignment

# Para "corregir" un valor se construye una tupla nueva
evento_corregido = evento[:2] + (22.0,)
print("Original: ", evento)
print("Corregido:", evento_corregido)

un_campo = ("sensor_07",)    # con coma: tupla de un elemento
sin_coma = ("sensor_07")     # sin coma: es solo un str entre paréntesis
print(type(un_campo).__name__, type(sin_coma).__name__)

Evento completo: ('2026-10-01T08:15:00', 'sensor_07', 21.4)
Sensor: sensor_07
Campos: 3
sensor_07 midió 21.4 ºC a las 08:15
Original:  ('2026-10-01T08:15:00', 'sensor_07', 21.4)
Corregido: ('2026-10-01T08:15:00', 'sensor_07', 22.0)
tuple str


**Paso a paso**

1. Se accede por índice igual que en una lista, y `len()` funciona igual.
2. El **desempaquetado** asigna cada campo a una variable. Debe haber tantas variables como elementos.
3. `momento[11:16]` extrae la hora (`08:15`) del texto con formato ISO.
4. Intentar asignar `evento[2] = 22.0` produce un `TypeError`. Para “corregir” un valor se construye una tupla nueva concatenando trozos: `evento[:2] + (22.0,)`.
5. Una tupla de un solo elemento necesita la coma final. Sin la coma, los paréntesis solo agrupan y el resultado es un `str`.


#### Ejemplo 2.2 · Lista de tuplas: lecturas de varios sensores

In [0]:
from operator import itemgetter

lecturas = [
    ("sensor_01", "Madrid", 21.4),
    ("sensor_02", "Sevilla", 31.8),
    ("sensor_03", "Bilbao", 17.2),
    ("sensor_04", "Sevilla", 29.5),
    ("sensor_05", "Madrid", 24.9),
]

# Desempaquetado directo en el for
for sensor, ciudad, temp in lecturas:
    if temp > 25:
        print("ALERTA", sensor, ciudad, temp)

ordenadas = sorted(lecturas, key=itemgetter(2))
print("Más fría:  ", ordenadas[0])
print("Más cálida:", ordenadas[-1])

# Ordenar por ciudad y, dentro de cada ciudad, por temperatura
for fila in sorted(lecturas, key=itemgetter(1, 2)):
    print(fila)

ALERTA sensor_02 Sevilla 31.8
ALERTA sensor_04 Sevilla 29.5
Más fría:   ('sensor_03', 'Bilbao', 17.2)
Más cálida: ('sensor_02', 'Sevilla', 31.8)
('sensor_03', 'Bilbao', 17.2)
('sensor_01', 'Madrid', 21.4)
('sensor_05', 'Madrid', 24.9)
('sensor_04', 'Sevilla', 29.5)
('sensor_02', 'Sevilla', 31.8)


**Paso a paso**

1. `for sensor, ciudad, temp in lecturas` desempaqueta cada tupla directamente en el bucle, sin usar índices.
2. `itemgetter(2)` ordena por temperatura.
3. `itemgetter(1, 2)` ordena por **dos criterios**: primero ciudad y, si hay empate, temperatura. Funciona porque las tuplas se comparan elemento a elemento, de izquierda a derecha.

####Ejemplo 2.3 · Desempaquetado extendido y divmod()

In [0]:
particiones = ("2026-09-26", "2026-09-27", "2026-09-28", "2026-09-29", "2026-09-30")

primera, *intermedias, ultima = particiones
print("Desde:", primera, "hasta:", ultima)
print("Intermedias:", intermedias)

# Duración de un job en segundos -> horas, minutos y segundos
duracion = 7384
horas, resto = divmod(duracion, 3600)
minutos, segundos = divmod(resto, 60)
print(f"El job tardó{horas} h{minutos} min{segundos} s")

# Intercambio de variables en una sola línea
origen, destino = "staging", "produccion"
origen, destino = destino, origen
print(origen, destino)

Desde: 2026-09-26 hasta: 2026-09-30
Intermedias: ['2026-09-27', '2026-09-28', '2026-09-29']
El job tardó2 h3 min4 s
produccion staging


**Paso a paso**

1. `primera, *intermedias, ultima` asigna el primer y el último elemento a sus variables. Lo que queda en medio va a `intermedias`, que siempre es una **lista**.
2. `divmod(a, b)` devuelve una tupla `(cociente, resto)`. 7384 s entre 3600 da 2 h y sobran 184 s. Esos 184 s son 3 min y 4 s.
3. `origen, destino = destino, origen` funciona porque Python primero crea la tupla `(destino, origen)` y después la desempaqueta. No hace falta una variable auxiliar.



####Ejemplo 2.4 · De columnas a filas y de filas a columnas con zip()

In [0]:
ids = [101, 102, 103, 104]
clientes = ["Acme", "Globex", "Initech", "Umbrella"]
importes = [1200.0, 340.5, 980.0, 15.0]

# De columnas a filas: cada fila es una tupla
filas = list(zip(ids, clientes, importes))
print("Filas:")
for fila in filas:
    print(" ", fila)

# De filas a columnas: zip(*filas) desempaqueta las filas
col_ids, col_clientes, col_importes = zip(*filas)
print("Columna clientes:", col_clientes)
print("Total importes:", sum(col_importes))

Filas:
  (101, 'Acme', 1200.0)
  (102, 'Globex', 340.5)
  (103, 'Initech', 980.0)
  (104, 'Umbrella', 15.0)
Columna clientes: ('Acme', 'Globex', 'Initech', 'Umbrella')
Total importes: 2535.5


**Paso a paso**

1. `zip(ids, clientes, importes)` toma un elemento de cada lista en cada vuelta y los junta en una tupla.
2. `zip(*filas)` hace la operación inversa. El  pasa cada fila como un argumento separado, y `zip` agrupa los primeros elementos de todas las filas, después los segundos, etc.
3. Cada columna que devuelve `zip` es una tupla, por eso `col_clientes` aparece entre paréntesis.

#### Ejemplo 2.5 · El modelo MapReduce con tuplas clave-valor

MapReduce, el modelo de procesamiento de Hadoop, trabaja siempre con **pares clave-valor**, y en Python un par es una tupla. Vamos a simular un conteo de palabras sobre líneas de log en tres fases.

In [0]:
from operator import itemgetter

# Cada línea la procesaría un nodo distinto del clúster
splits = [
    "ERROR disco lleno en datanode3",
    "WARN latencia alta en datanode1",
    "ERROR timeout en datanode1",
]

# MAP: cada palabra se emite como una tupla (clave, 1)
pares = []
for linea in splits:
    for palabra in linea.split():
        pares.append((palabra, 1))
print("Pares emitidos:", len(pares))
print("Primeros pares:", pares[:3])

# SHUFFLE / SORT: al ordenar por clave, las tuplas iguales quedan juntas
pares.sort(key=itemgetter(0))

# REDUCE: sumar los valores de cada grupo de claves consecutivas
resultado = []
clave_actual, suma = pares[0]
for clave, valor in pares[1:]:
    if clave == clave_actual:
        suma += valor
    else:
        resultado.append((clave_actual, suma))
        clave_actual, suma = clave, valor
resultado.append((clave_actual, suma))

print("Claves distintas:", len(resultado))
for clave, total in resultado:
    if total > 1:
        print(clave, total)

Pares emitidos: 14
Primeros pares: [('ERROR', 1), ('disco', 1), ('lleno', 1)]
Claves distintas: 10
ERROR 2
datanode1 2
en 3


**Paso a paso**

1. **Map:** cada nodo procesa su trozo de datos (*split*) y emite una tupla `(palabra, 1)` por cada palabra, sin contar nada todavía.
2. **Shuffle / sort:** el sistema ordena los pares por clave para que las tuplas de una misma palabra queden juntas. En un clúster, esta fase mueve datos por la red entre nodos y es la más costosa.
3. **Reduce:** se recorren los pares ordenados. Mientras la clave se repite, se acumula; cuando cambia, se guarda el total de la clave anterior y se empieza a contar la nueva.
4. El último grupo se añade después del bucle, porque no hay ninguna clave nueva que provoque su guardado.
5. Python ordena las mayúsculas antes que las minúsculas, por eso `ERROR` aparece primero.

#### Ejemplo 2.6 · Calcular el lag de un consumidor de Kafka

Kafka identifica cada mensaje por tres datos: el topic, la partición y su posición dentro de ella (*offset*). El **lag** es el número de mensajes que el consumidor todavía no ha procesado.

In [0]:
from operator import itemgetter

# (topic, partición, offset): último mensaje escrito en el broker
ultimos = [("pedidos", 0, 15230), ("pedidos", 1, 14980), ("pedidos", 2, 15410)]
# (topic, partición, offset): último mensaje confirmado por el consumidor
confirmados = [("pedidos", 0, 15228), ("pedidos", 1, 13100), ("pedidos", 2, 15410)]

lags = []
for (topic, particion, fin), (_, _, leido) in zip(ultimos, confirmados):
    lags.append((topic, particion, fin - leido))

for lag in sorted(lags, key=itemgetter(2), reverse=True):
    print(lag)

total = 0
for _, _, pendientes in lags:
    total += pendientes
print("Lag total del consumidor:", total)
print("Partición más retrasada:", max(lags, key=itemgetter(2))[:2])

('pedidos', 1, 1880)
('pedidos', 0, 2)
('pedidos', 2, 0)
Lag total del consumidor: 1882
Partición más retrasada: ('pedidos', 1)


**Paso a paso**

1. `zip()` empareja la tupla de cada partición en las dos listas. Funciona porque ambas listas están en el mismo orden de partición.
2. En el `for` se desempaquetan **dos tuplas a la vez**. El guion bajo `_` indica los campos que no necesitamos.
3. Lag = offset escrito − offset confirmado. La partición 1 acumula 1880 mensajes pendientes: el consumidor asignado a ella va retrasado.
4. `max(...)[:2]` toma la tupla con más lag y se queda con sus dos primeros campos, topic y partición.

### Ejercicios propuestos

#### Ejercicio 2.1 · Resumen de pedidos


In [0]:
# (pedido_id, cliente, importe, estado)
pedidos = [
    (5001, "Acme", 250.0, "ENTREGADO"),
    (5002, "Globex", 1340.0, "ENTREGADO"),
    (5003, "Initech", 89.9, "CANCELADO"),
    (5004, "Acme", 610.0, "ENTREGADO"),
    (5005, "Umbrella", 45.0, "CANCELADO"),
    (5006, "Globex", 300.0, "PENDIENTE"),
]
total = 0
for pe in pedidos:
    if pe[3] == "CANCELADO" or pe[3]=="PENDIENTE":
        continue
    else:
        total += pe[2]

print("Total:", total)

print(f"pedido de mayor importe {max(pedidos, key=itemgetter(2))[0]}")

print(f"Cancelados {([pe[1:3] for pe in pedidos if pe[3] == 'CANCELADO'])}")

estados = [pe[3] for pe in pedidos]

total_entregados = estados.count('ENTREGADO')
total_cancelados = estados.count('CANCELADO')
total_pendientes = estados.count('PENDIENTE')
# 3. Mostramos el resultado
print(f"pedidos de cada tipo --> Entregado: {total_entregados}; Cancelado: {total_cancelados} ; Pendiente: {total_pendientes}")


Total: 2200.0
pedido de mayor importe 5002
Cancelados [('Initech', 89.9), ('Umbrella', 45.0)]
pedidos de cada tipo --> Entregado: 3; Cancelado: 2 ; Pendiente: 1


#### Ejercicio 2.2 · Validar filas contra un esquema

El esquema de una tabla se describe como una tupla de tuplas `(columna, tipo)`. Comprueba cada fila con `isinstance()` e informa de cada valor que no tenga el tipo esperado.
Pistas: enumerate(filas, start=1) numera las filas desde 1, y zip(esquema, fila) empareja cada columna del esquema con su valor. El nombre de un tipo se obtiene con tipo.__name__.

In [0]:
esquema = (("id", int), ("cliente", str), ("importe", float), ("activo", bool))
filas = [
    (1, "Acme", 120.5, True),
    (2, "Globex", "340", True),
    ("3", "Initech", 99.9, False),
    (4, "Umbrella", 15.0, "no"),
]

total_errores = 0

for num_fila, fila in enumerate(filas, start=1):
    for (columna, tipo_esperado), valor in zip(esquema, fila):
        if not isinstance(valor, tipo_esperado):
            tipo_recibido = type(valor).__name__
            print(
                f"Fila {num_fila}: '{columna}' esperaba {tipo_esperado.__name__} y recibió {tipo_recibido}"
            )
            total_errores += 1

print(f"Filas: {len(filas)} | Errores: {total_errores}")


Fila 2: 'importe' esperaba float y recibió str
Fila 3: 'id' esperaba int y recibió str
Fila 4: 'activo' esperaba bool y recibió str
Filas: 4 | Errores: 3


#### Ejercicio 2.3 · Comparar versiones de un esquema

Un registro de esquemas guarda versiones como texto. Compararlas como texto da resultados erróneos, porque `"1.10.0"` se considera menor que `"1.9.7"`.
1. Muestra el resultado incorrecto al comparar y ordenar como texto.
2. Convierte cada versión en una tupla de enteros `(mayor, menor, parche)`.
3. Ordénalas, obtén la última y muéstrala con el formato `v2.0.0`.

In [0]:
versiones = ["1.4.2", "1.10.0", "1.9.7", "2.0.0", "1.10.3"]

print(f"Como texto, ¿1.10.0 > 1.9.7? {'1.10.0' > '1.9.7'}")

ordenado_como_texto = sorted(versiones)
print(f"Ordenadas como texto: {ordenado_como_texto}")



lista_tuplas = [tuple(int(x) for x in v.split(".")) for v in versiones]
print(f"Como tuplas: {lista_tuplas}")



ordenadas = sorted(lista_tuplas)
print(f"Ordenadas: {ordenadas}")

ultima_tupla = ordenadas[-1]

ultima_version_str = ".".join(str(x) for x in ultima_tupla)
print(f"Última versión: v{ultima_version_str}")

v_1_10_0 = (1, 10, 0)
v_1_9_7 = (1, 9, 7)
print(f"¿1.10.0 es posterior a 1.9.7? {v_1_10_0 > v_1_9_7}")


Como texto, ¿1.10.0 > 1.9.7? False
Ordenadas como texto: ['1.10.0', '1.10.3', '1.4.2', '1.9.7', '2.0.0']
Como tuplas: [(1, 4, 2), (1, 10, 0), (1, 9, 7), (2, 0, 0), (1, 10, 3)]
Ordenadas: [(1, 4, 2), (1, 9, 7), (1, 10, 0), (1, 10, 3), (2, 0, 0)]
Última versión: v2.0.0
¿1.10.0 es posterior a 1.9.7? True


#### Ejercicio 2.4 · Detectar solapes entre ejecuciones de jobs

Cada tupla indica el minuto del día en que arranca y termina un job. Dos jobs que comparten clúster no deberían solaparse.

Ordena las ejecuciones por hora de inicio. Después compara cada ejecución con la siguiente: hay solape si la siguiente arranca antes de que termine la actual. Para cada solape, muestra los jobs implicados y los minutos de solape.

In [0]:
from operator import itemgetter

ejecuciones = [
    ("job_c", 130, 170),
    ("job_a", 60, 95),
    ("job_d", 165, 200),
    ("job_b", 90, 120),
]

ejecuciones_ordenadas = sorted(ejecuciones, key=itemgetter(1))

nombres_ordenados = [job[0] for job in ejecuciones_ordenadas]
print(f"Orden de arranque: {nombres_ordenados}")

solapes_contados = 0
for i in range(len(ejecuciones_ordenadas) - 1):
    job_actual, _, fin_actual = ejecuciones_ordenadas[i]
    job_siguiente, inicio_siguiente, _ = ejecuciones_ordenadas[i + 1]

    if inicio_siguiente < fin_actual:
        minutos_solape = fin_actual - inicio_siguiente
        solapes_contados += 1
        print(
            f"SOLAPE: {job_siguiente} arranca en {inicio_siguiente} y {job_actual} termina en {fin_actual} ({minutos_solape} min)"
        )

print(f"Solapes detectados: {solapes_contados}")


Orden de arranque: ['job_a', 'job_b', 'job_c', 'job_d']
SOLAPE: job_b arranca en 90 y job_a termina en 95 (5 min)
SOLAPE: job_d arranca en 165 y job_c termina en 170 (5 min)
Solapes detectados: 2


#### Ejercicio 2.5 · MapReduce sobre logs de acceso

Cada línea de log tiene el formato `método ruta código_http bytes`.
1. **Map:** emite una tupla `(código, bytes)` por línea. Los bytes deben ser `int`.
2. **Shuffle / sort:** ordena los pares por código.
3. **Reduce:** obtén una lista de tuplas `(código, peticiones, bytes_totales)` y muéstrala.

In [0]:
from collections import defaultdict

logs = [
    "GET /pedidos 200 5120",
    "GET /clientes 404 310",
    "POST /pedidos 201 880",
    "GET /pedidos 200 4980",
    "GET /stock 500 120",
    "GET /clientes 200 2210",
    "GET /stock 500 95",
    "POST /pedidos 201 910",
]


def map_linea(linea):
    partes = linea.split()
    codigo_http = partes[2]
    bytes_int = int(partes[3])
    return (codigo_http, bytes_int)

fase_map = list(map(map_linea, logs))
print(f"Fase map: {fase_map}\n")



agrupados = defaultdict(list)
for codigo, bytes_req in fase_map:
    agrupados[codigo].append(bytes_req)

fase_shuffle_sort = sorted(agrupados.items())


resultado_final = []
for codigo, lista_bytes in fase_shuffle_sort:
    peticiones = len(lista_bytes)
    bytes_totales = sum(lista_bytes)
    resultado_final.append((codigo, peticiones, bytes_totales))
    
    print(f"{codigo}: {peticiones} peticiones, {bytes_totales} bytes")


Fase map: [('200', 5120), ('404', 310), ('201', 880), ('200', 4980), ('500', 120), ('200', 2210), ('500', 95), ('201', 910)]

200: 3 peticiones, 12310 bytes
201: 2 peticiones, 1790 bytes
404: 1 peticiones, 310 bytes
500: 2 peticiones, 215 bytes


#### Ejercicio 2.6 · *Partition pruning* con tuplas de fecha

Cuando una consulta filtra por fecha sobre una tabla particionada, el motor solo lee las carpetas de las fechas pedidas y descarta el resto sin abrirlas. Esto se llama *partition pruning*.
1. Selecciona las particiones cuya fecha `(año, mes, día)` esté entre `desde` y `hasta`, ambos incluidos. Compara tuplas directamente, sin convertirlas a texto.
2. Muestra la ruta de cada partición seleccionada con el formato `ventas/year=AAAA/month=MM/day=DD` y su número de ficheros.
3. Indica cuántos ficheros se leen del total y qué porcentaje se ha descartado.

In [0]:
particiones = [
    (2026, 9, 27, 14),
    (2026, 9, 28, 12),
    (2026, 9, 29, 15),
    (2026, 9, 30, 13),
    (2026, 10, 1, 16),
    (2026, 10, 2, 11),
    (2026, 10, 3, 12),
]

desde = (2026, 9, 29)
hasta = (2026, 10, 1)

total_ficheros = sum(p[3] for p in particiones)
ficheros_leidos = 0

for anio, mes, dia, ficheros in particiones:
    fecha_actual = (anio, mes, dia)
    
    if desde <= fecha_actual <= hasta:
        ficheros_leidos += ficheros
        print(f"ventas/year={anio}/month={mes:02d}/day={dia:02d} -> {ficheros} ficheros")

ficheros_descartados = total_ficheros - ficheros_leidos
porcentaje_descartado = (ficheros_descartados / total_ficheros) * 100

print(f"Ficheros leídos: {ficheros_leidos} de {total_ficheros}")
print(f"Ficheros descartados: {porcentaje_descartado:.1f} %")


ventas/year=2026/month=09/day=29 -> 15 ficheros
ventas/year=2026/month=09/day=30 -> 13 ficheros
ventas/year=2026/month=10/day=01 -> 16 ficheros
Ficheros leídos: 44 de 93
Ficheros descartados: 52.7 %


## Sección 3. Set

Un `set` es una colección **sin orden y sin duplicados**. Sus dos usos principales en ingeniería de datos son **deduplicar** y **comparar colecciones**: qué registros faltan, cuáles sobran y cuáles coinciden. Además, comprobar si un elemento pertenece a un `set` es mucho más rápido que buscarlo en una lista.

> Como un `set` no tiene orden, los ejemplos usan `sorted()` para mostrarlos. Así la salida es siempre la misma.
> 

### Ejemplos resueltos

#### Ejemplo 3.1 · Clientes únicos de dos fuentes

In [0]:
crm = ["C001", "C002", "C003", "C002", "C004", "C001"]
ecommerce = ["C003", "C005", "C004", "C006", "C005"]

print("Registros en CRM:", len(crm), "| únicos:", len(set(crm)))

clientes = set(crm)
clientes.update(ecommerce)      # añade todos los elementos de la otra fuente
print("Clientes únicos totales:", len(clientes))
print(sorted(clientes))

clientes.add("C007")
clientes.add("C001")            # ya existe: no cambia nada
clientes.discard("C999")        # no existe: discard() no da error
print("Tras las altas:", len(clientes))

Registros en CRM: 6 | únicos: 4
Clientes únicos totales: 6
['C001', 'C002', 'C003', 'C004', 'C005', 'C006']
Tras las altas: 7


**Paso a paso**

1. `set(crm)` elimina automáticamente los duplicados: 6 registros se quedan en 4 clientes.
2. `update()` añade todos los elementos de otra colección. Los repetidos se ignoran.
3. `add()` añade un elemento. Si ya existía, no pasa nada.
4. `discard()` elimina un elemento sin dar error si no existe. `remove()`, en cambio, lanzaría `KeyError`.

#### Ejemplo 3.2 · Reconciliación entre origen y destino

Después de una carga, hay que comprobar que todos los ids del sistema origen han llegado al data warehouse.

In [0]:
origen = {101, 102, 103, 104, 105, 106}
destino = {101, 102, 104, 106, 107}

print("Cargados correctamente:", sorted(origen & destino))
print("Faltan en destino:     ", sorted(origen - destino))
print("Sobran en destino:     ", sorted(destino - origen))
print("Discrepancias totales: ", sorted(origen ^ destino))
print("¿Carga completa?", origen <= destino)

Cargados correctamente: [101, 102, 104, 106]
Faltan en destino:      [103, 105]
Sobran en destino:      [107]
Discrepancias totales:  [103, 105, 107]
¿Carga completa? False


**Paso a paso**

| Operador | Operación | Pregunta que responde |
| --- | --- | --- |
| `a & b` | Intersección | ¿Qué está en los dos? |
| `a - b` | Diferencia | ¿Qué está en `a` pero no en `b`? |
| `a ^ b` | Diferencia simétrica | ¿Qué está solo en uno de los dos? |
| `a <= b` | Subconjunto | ¿Está todo `a` dentro de `b`? |

Los ids 103 y 105 no se cargaron, y el 107 está en destino aunque no viene del origen. Puede ser un registro antiguo que debería haberse borrado.

####Ejemplo 3.3 · Validar las columnas de un fichero recibido

In [0]:
esperadas = {"id_pedido", "fecha", "cliente", "importe", "moneda"}
recibidas = ["id_pedido", "fecha", "importe", "cliente", "canal", "importe"]

recibidas_set = set(recibidas)
print("¿Columnas duplicadas en origen?", len(recibidas) != len(recibidas_set))
print("Faltan:", sorted(esperadas - recibidas_set))
print("Nuevas:", sorted(recibidas_set - esperadas))
print("¿Esquema válido?", esperadas.issubset(recibidas_set))

¿Columnas duplicadas en origen? True
Faltan: ['moneda']
Nuevas: ['canal']
¿Esquema válido? False


**Paso a paso**

1. Si al convertir la lista en `set` se reduce su tamaño, había columnas duplicadas: `importe` aparece dos veces.
2. `esperadas - recibidas_set` indica qué columnas faltan, y la resta inversa indica las columnas nuevas.
3. `issubset()` es la versión con método del operador `<=`.

#### Ejemplo 3.4 · Filtrar logs con una lista negra de IPs

In [0]:
ips_bloqueadas = {"10.0.0.13", "192.168.1.50", "172.16.4.2"}

accesos = [
    ("08:01", "10.0.0.7"),
    ("08:02", "192.168.1.50"),
    ("08:02", "10.0.0.8"),
    ("08:03", "172.16.4.2"),
    ("08:05", "192.168.1.50"),
]

sospechosos = []
ips_vistas = set()
for hora, ip in accesos:
    ips_vistas.add(ip)
    if ip in ips_bloqueadas:          # búsqueda muy rápida en un set
        sospechosos.append((hora, ip))

print("Accesos bloqueados:", len(sospechosos))
for s in sospechosos:
    print(" ", s)
print("IPs distintas vistas:", len(ips_vistas))
print("Bloqueadas que no han aparecido:", ips_bloqueadas - ips_vistas)

Accesos bloqueados: 3
  ('08:02', '192.168.1.50')
  ('08:03', '172.16.4.2')
  ('08:05', '192.168.1.50')
IPs distintas vistas: 4
Bloqueadas que no han aparecido: {'10.0.0.13'}


**Paso a paso**

1. `ip in ips_bloqueadas` es casi instantáneo aunque el `set` tenga millones de IPs, porque internamente se busca por *hash*. En una lista habría que revisar los elementos uno a uno.
2. Un `set` puede contener tuplas, porque son inmutables, pero no listas. `{[1, 2]}` produce `TypeError: unhashable type: 'list'`.
3. Al restar las IPs vistas de las bloqueadas sabemos qué IPs de la lista negra no han aparecido en este log.

#### Ejemplo 3.5 · Contar usuarios únicos en datos particionados

Contar elementos distintos (`COUNT(DISTINCT ...)`) es una de las operaciones más delicadas en un sistema distribuido, porque un mismo usuario puede aparecer en varias particiones.

In [0]:
# Usuarios que generaron eventos, repartidos en 3 particiones
particion_0 = ["u1", "u2", "u3", "u2", "u4"]
particion_1 = ["u3", "u5", "u1", "u6"]
particion_2 = ["u2", "u7", "u5", "u5"]

unicos_0 = set(particion_0)
unicos_1 = set(particion_1)
unicos_2 = set(particion_2)
print("Únicos por partición:", len(unicos_0), len(unicos_1), len(unicos_2))

suma_parcial = len(unicos_0) + len(unicos_1) + len(unicos_2)
print("Suma de los únicos parciales:", suma_parcial, "(incorrecto)")

globales = unicos_0 | unicos_1 | unicos_2
print("Únicos globales:", len(globales), sorted(globales))
print("Usuarios contados de más:", suma_parcial - len(globales))

Únicos por partición: 4 4 3
Suma de los únicos parciales: 11 (incorrecto)
Únicos globales: 7 ['u1', 'u2', 'u3', 'u4', 'u5', 'u6', 'u7']
Usuarios contados de más: 4


**Paso a paso**

1. Cada partición calcula su `set` de usuarios únicos de forma independiente, como haría cada nodo del clúster.
2. **Sumar los tamaños de los `set` parciales es un error**: `u1`, `u2`, `u3` y `u5` aparecen en más de una partición y se cuentan varias veces.
3. El resultado correcto exige **unir** los `set` (`|`) y contar después. En un clúster, esto obliga a mover los identificadores por la red hasta un mismo nodo.
4. Con miles de millones de usuarios, unir los `set` exactos es muy caro. Por eso los motores de Big Data ofrecen conteos aproximados, como `approx_count_distinct()` en Spark.

#### Ejemplo 3.6 · Eliminar duplicados en un stream con entrega *at-least-once*

Muchos sistemas de mensajería garantizan que cada evento llega **al menos una vez** (*at-least-once*). Si hay un fallo, el evento se reenvía, así que el consumidor debe descartar los repetidos para no escribirlos dos veces.

In [0]:
procesados = set()        # ids de eventos ya escritos en destino

micro_lotes = [
    ["e01", "e02", "e03"],
    ["e03", "e04", "e05"],          # e03 se reenvió tras un fallo
    ["e05", "e06", "e02", "e07"],   # e05 y e02 llegan otra vez
]

for num, lote in enumerate(micro_lotes, start=1):
    nuevos = []
    for evento in lote:
        if evento not in procesados:
            nuevos.append(evento)
            procesados.add(evento)
    descartados = len(lote) - len(nuevos)
    print(f"Lote{num}: recibidos{len(lote)}, escritos{nuevos}, descartados{descartados}")

print("Eventos escritos en total:", len(procesados))

Lote1: recibidos3, escritos['e01', 'e02', 'e03'], descartados0
Lote2: recibidos3, escritos['e04', 'e05'], descartados1
Lote3: recibidos4, escritos['e06', 'e07'], descartados2
Eventos escritos en total: 7


**Paso a paso**

1. `procesados` es un `set` que se mantiene entre micro-lotes y recuerda qué eventos ya se han escrito.
2. Para cada evento, `in` comprueba de forma inmediata si ya se procesó. Si es nuevo, se escribe y se añade al `set`.
3. `enumerate(..., start=1)` numera los lotes desde 1.
4. Gracias a esta comprobación, procesar varias veces el mismo evento produce el mismo resultado. Esta propiedad se llama **idempotencia**.
5. En un sistema real este `set` no puede crecer sin límite. Spark Structured Streaming, por ejemplo, olvida los ids más antiguos que un tiempo configurado (*watermark*).

### Ejercicios propuestos

#### Ejercicio 3.1 · Usuarios activos y retención
Calcula, solo con operadores de `set`:

- Los usuarios activos los tres días.
- Cuántos usuarios distintos estuvieron activos al menos un día.
- Los que solo entraron el lunes.
- Los usuarios nuevos del miércoles, es decir, los que no estuvieron ni el lunes ni el martes.
- La retención del lunes al martes: porcentaje de usuarios del lunes que volvieron el martes.

In [0]:
lunes = {"u01", "u02", "u03", "u04", "u05", "u06"}
martes = {"u02", "u03", "u05", "u07", "u08"}
miercoles = {"u01", "u03", "u05", "u08", "u09"}

print("Usuarios que han entrado en los 3 días:", lunes & martes & miercoles)
print("Activos al menos un dia:",len( lunes | martes | miercoles))
print("Solo el lunes:", lunes - (martes | miercoles))
print("Solo el miercoles:", miercoles - (lunes | martes))
print("Retencion de lunes a martes ", len(lunes & martes) / len(lunes) * 100,"%")


Usuarios que han entrado en los 3 días: {'u03', 'u05'}
Activos al menos un dia: 9
Solo el lunes: {'u04', 'u06'}
Solo el miercoles: {'u09'}
Retencion de lunes a martes  50.0 %


#### Ejercicio 3.2 · Detectar datos personales antes de publicar

Antes de publicar datasets en una zona compartida, el equipo de gobierno del dato pide revisar si contienen columnas con información personal (PII).
Para cada dataset, usa isdisjoint() para saber si está libre de PII. Si no lo está, muestra qué columnas sensibles contiene. Al final, indica cuántos datasets necesitan enmascarado.


In [0]:
pii = {"dni", "email", "telefono", "iban", "fecha_nacimiento"}

datasets = [
    ("ventas", {"id_venta", "fecha", "importe", "id_cliente"}),
    ("clientes", {"id_cliente", "nombre", "email", "telefono", "ciudad"}),
    ("pagos", {"id_pago", "iban", "importe", "fecha"}),
    ("logs_web", {"timestamp", "url", "user_agent"}),
]

contador_enmascarado = 0

for nombre, columnas in datasets:
    if columnas.isdisjoint(pii):
        print(f"{nombre:<10} sin PII")
    else:
        campos_encontrados = list(columnas & pii)
        print(f"{nombre:<10} PII -> {campos_encontrados}")
        contador_enmascarado += 1

print(f"Datasets que requieren enmascarado: {contador_enmascarado}")


ventas     sin PII
clientes   PII -> ['telefono', 'email']
pagos      PII -> ['iban']
logs_web   sin PII
Datasets que requieren enmascarado: 2


#### Ejercicio 3.3 · ¿Qué transacciones vienen repetidas?

`set(transacciones)` elimina los duplicados, pero no dice **cuáles** estaban repetidos. Usa dos `set`, `vistos` y `repetidos`, en un solo recorrido.

In [0]:
transacciones = ["T10", "T11", "T12", "T11", "T13", "T14", "T12", "T11", "T15"]

vistos = set()
repetidos = set()

for t in transacciones:
    if t in vistos:
        repetidos.add(t)
    else:
        vistos.add(t)

lista_repetidos = sorted(list(repetidos))
unicos_totales = len(vistos)

print(f"Repetidos: {lista_repetidos}")
print(f"Únicos: {unicos_totales} de {len(transacciones)}")


Repetidos: ['T11', 'T12']
Únicos: 6 de 9


#### Ejercicio 3.4 · Rutas sin vuelta en un log de vuelos
1. Crea un `set` de rutas distintas. Una ruta es una tupla `(origen, destino)`.
2. Obtén las rutas que no tienen vuelta, es decir, aquellas en las que `(destino, origen)` no está en el `set`.
3. Incorpora las rutas de mañana con el operador `|=` y repite el cálculo.

In [0]:
rutas_hoy = [("MAD", "BCN"), ("BCN", "MAD"), ("MAD", "LIS"), ("MAD", "BCN"),
             ("SVQ", "MAD"), ("LIS", "MAD"), ("BIO", "PMI")]
rutas_manana = [("PMI", "BIO"), ("MAD", "VLC")]

set_hoy = set(rutas_hoy)
print(f"Rutas distintas hoy: {len(set_hoy)}")

sin_vuelta_hoy = sorted([r for r in set_hoy if (r[1], r[0]) not in set_hoy])
print(f"Sin vuelta: {sin_vuelta_hoy}")

set_hoy |= set(rutas_manana)
print(f"Rutas acumuladas: {len(set_hoy)}")

sin_vuelta_manana = sorted([r for r in set_hoy if (r[1], r[0]) not in set_hoy])
print(f"Sin vuelta tras añadir mañana: {sin_vuelta_manana}")


Rutas distintas hoy: 6
Sin vuelta: [('BIO', 'PMI'), ('SVQ', 'MAD')]
Rutas acumuladas: 8
Sin vuelta tras añadir mañana: [('MAD', 'VLC'), ('SVQ', 'MAD')]


#### Ejercicio 3.5 · Particiones que hay que reprocesar

Un job diario escribe una carpeta por fecha en HDFS y, al terminar con éxito, deja dentro un fichero vacío llamado `_SUCCESS`. Una carpeta sin ese marcador puede tener datos a medias.
Usando solo operaciones de `set`, obtén:

- Las particiones esperadas que no existen.
- Las particiones esperadas que existen pero no tienen `_SUCCESS`.
- La lista de particiones a reprocesar (unión de las dos anteriores).
- Las carpetas que existen pero no corresponden al rango esperado.
- Cuántas particiones esperadas están completas y qué porcentaje representan.

In [0]:
esperadas = {"2026-10-01", "2026-10-02", "2026-10-03", "2026-10-04",
             "2026-10-05", "2026-10-06", "2026-10-07"}
presentes = {"2026-09-30", "2026-10-01", "2026-10-02", "2026-10-03",
             "2026-10-05", "2026-10-06", "2026-10-07"}
con_success = {"2026-09-30", "2026-10-01", "2026-10-02", "2026-10-05", "2026-10-07"}

faltan = esperadas - presentes
incompletas = (esperadas & presentes) - con_success
a_reprocesar = faltan | incompletas
fuera_de_rango = presentes - esperadas

completas = esperadas & con_success
porcentaje = (len(completas) / len(esperadas)) * 100

print(f"Faltan: {sorted(list(faltan))}")
print(f"Incompletas: {sorted(list(incompletas))}")
print(f"A reprocesar: {sorted(list(a_reprocesar))}")
print(f"Fuera de rango: {sorted(list(fuera_de_rango))}")
print(f"Completas: {len(completas)} de {len(esperadas)} ({porcentaje:.1f} %)")


Faltan: ['2026-10-04']
Incompletas: ['2026-10-03', '2026-10-06']
A reprocesar: ['2026-10-03', '2026-10-04', '2026-10-06']
Fuera de rango: ['2026-09-30']
Completas: 4 de 7 (57.1 %)


#### Ejercicio 3.6 · Réplicas vivas de los bloques de HDFS

Dos DataNodes del clúster han dejado de responder. El NameNode necesita saber qué bloques han perdido réplicas.
1. Para cada bloque, calcula sus réplicas vivas (intersección con `nodos_vivos`) y muéstralas ordenadas.
2. Reúne en un `set` todos los nodos que aparecen en los bloques (usa `|=`) y obtén los nodos caídos.
3. Clasifica los bloques: **en riesgo** si les queda 1 réplica viva y **perdidos** si no les queda ninguna.

In [0]:
nodos_vivos = {"dn1", "dn2", "dn4", "dn5"}

bloques = [
    ("blk_001", {"dn1", "dn2", "dn3"}),
    ("blk_002", {"dn3", "dn6", "dn4"}),
    ("blk_003", {"dn2", "dn4", "dn5"}),
    ("blk_004", {"dn3", "dn6", "dn1"}),
    ("blk_005", {"dn6", "dn3", "dn2"}),
    ("blk_006", {"dn3", "dn6"}),
]

todos_los_nodos = set()
en_riesgo = []
perdidos = []

for id_bloque, nodos in bloques:
    vivas = nodos & nodos_vivos
    lista_vivas = sorted(list(vivas))
    
    print(f"{id_bloque}: {len(vivas)} réplicas vivas {lista_vivas}")
    
    todos_los_nodos |= nodos
    
    if len(vivas) == 1:
        en_riesgo.append(id_bloque)
    elif len(vivas) == 0:
        perdidos.append(id_bloque)

nodos_caidos = sorted(list(todos_los_nodos - nodos_vivos))

print(f"Nodos caídos: {nodos_caidos}")
print(f"Bloques en riesgo: {en_riesgo}")
print(f"Bloques perdidos: {perdidos}")


blk_001: 2 réplicas vivas ['dn1', 'dn2']
blk_002: 1 réplicas vivas ['dn4']
blk_003: 3 réplicas vivas ['dn2', 'dn4', 'dn5']
blk_004: 1 réplicas vivas ['dn1']
blk_005: 1 réplicas vivas ['dn2']
blk_006: 0 réplicas vivas []
Nodos caídos: ['dn3', 'dn6']
Bloques en riesgo: ['blk_002', 'blk_004', 'blk_005']
Bloques perdidos: ['blk_006']


## Sección 4. Diccionarios

Un diccionario guarda pares **clave → valor** y permite acceder a cada valor por su clave de forma muy rápida. Es la estructura que corresponde a un objeto JSON. También es la herramienta básica para contar, agrupar y cruzar datos sin una base de datos.

### Ejemplos resueltos

#### Ejemplo 4.1 · Un pedido como registro JSON

In [0]:
pedido = {
    "id": 50231,
    "cliente": "Acme",
    "importe": 1250.0,
    "moneda": "EUR",
}

print(pedido["cliente"])
print(pedido.get("canal"))               # la clave no existe: None
print(pedido.get("canal", "online"))     # valor por defecto

pedido["canal"] = "tienda"      # añadir
pedido["importe"] = 1190.0      # modificar
moneda = pedido.pop("moneda")   # extraer y eliminar
print("Moneda retirada:", moneda)
print(pedido)

print("Claves:", list(pedido.keys()))
for campo, valor in pedido.items():
    print(f"{campo:<8} ->{valor}")
print("¿Tiene 'importe'?", "importe" in pedido)

Acme
None
online
Moneda retirada: EUR
{'id': 50231, 'cliente': 'Acme', 'importe': 1190.0, 'canal': 'tienda'}
Claves: ['id', 'cliente', 'importe', 'canal']
id       ->50231
cliente  ->Acme
importe  ->1190.0
canal    ->tienda
¿Tiene 'importe'? True


**Paso a paso**

1. `pedido["cliente"]` lanza `KeyError` si la clave no existe. `get()` devuelve `None` en ese caso, o el valor por defecto que le indiquemos.
2. Asignar a una clave nueva la **añade**, y asignar a una clave existente la **modifica**.
3. `pop()` elimina la clave y devuelve su valor.
4. `keys()`, `values()` e `items()` permiten recorrer las claves, los valores o los pares. Los diccionarios conservan el orden de inserción.
5. `in` comprueba si existe una **clave**, no un valor.

####Ejemplo 4.2 · Contar eventos por tipo

In [0]:
eventos = ["login", "click", "click", "compra", "login", "click",
           "error", "click", "logout", "error"]

conteo = {}
for e in eventos:
    conteo[e] = conteo.get(e, 0) + 1
print(conteo)

for tipo in sorted(conteo):
    print(f"{tipo:<7}{'#' * conteo[tipo]}{conteo[tipo]}")

{'login': 2, 'click': 4, 'compra': 1, 'error': 2, 'logout': 1}
click  ####4
compra #1
error  ##2
login  ##2
logout #1


**Paso a paso**

1. `conteo.get(e, 0)` devuelve el contador actual o 0 si es la primera vez que aparece el evento. Así no hace falta comprobar antes si la clave existe.
2. `sorted(conteo)` devuelve las **claves** ordenadas alfabéticamente.
3. `'#' * n` repite el carácter `n` veces y produce un histograma de texto.

####Ejemplo 4.3 · Agrupar y agregar ventas por tienda (un GROUP BY en Python)

In [0]:
ventas = [
    ("MAD01", 1250.0), ("BCN02", 980.5), ("MAD01", 1530.0),
    ("VLC03", 410.0), ("BCN02", 1105.0), ("MAD01", 300.0),
]

resumen = {}
for tienda, importe in ventas:
    if tienda not in resumen:
        resumen[tienda] = {"total": 0.0, "pedidos": 0}
    resumen[tienda]["total"] += importe
    resumen[tienda]["pedidos"] += 1

print(resumen)
for tienda, datos in resumen.items():
    media = datos["total"] / datos["pedidos"]
    print(f"{tienda}:{datos['pedidos']} pedidos, total{datos['total']:.2f}, media{media:.2f}")

{'MAD01': {'total': 3080.0, 'pedidos': 3}, 'BCN02': {'total': 2085.5, 'pedidos': 2}, 'VLC03': {'total': 410.0, 'pedidos': 1}}
MAD01:3 pedidos, total3080.00, media1026.67
BCN02:2 pedidos, total2085.50, media1042.75
VLC03:1 pedidos, total410.00, media410.00


**Paso a paso**

1. Cada tienda es una clave, y su valor es otro diccionario con dos acumuladores.
2. La primera vez que aparece una tienda se inicializan sus acumuladores. Las siguientes veces solo se actualizan.
3. Es lo mismo que haría en SQL `SELECT tienda, COUNT(*), SUM(importe), AVG(importe) ... GROUP BY tienda`.

#### Ejemplo 4.4 · Enriquecer pedidos con datos de clientes (un join por clave)

In [0]:
clientes = {
    "C001": {"nombre": "Acme", "pais": "ES"},
    "C002": {"nombre": "Globex", "pais": "PT"},
    "C003": {"nombre": "Initech", "pais": "MX"},
}
pedidos = [("P1", "C001", 300.0), ("P2", "C003", 125.5),
           ("P3", "C009", 80.0), ("P4", "C001", 47.0)]

desconocido = {"nombre": "DESCONOCIDO", "pais": "??"}
for id_pedido, id_cliente, importe in pedidos:
    cli = clientes.get(id_cliente, desconocido)
    print(id_pedido, id_cliente, cli["nombre"], cli["pais"], importe)

P1 C001 Acme ES 300.0
P2 C003 Initech MX 125.5
P3 C009 DESCONOCIDO ?? 80.0
P4 C001 Acme ES 47.0


**Paso a paso**

1. El diccionario `clientes` funciona como una **tabla de búsqueda** indexada por `id_cliente`. Así funciona internamente un *hash join*.
2. `get(id_cliente, desconocido)` resuelve los clientes que no existen, como `C009`, sin que el proceso falle. Es el equivalente a un `LEFT JOIN` con un valor por defecto.


#### Ejemplo 4.5 · Fusionar configuraciones de un pipeline

In [0]:
config_base = {
    "formato": "parquet",
    "compresion": "snappy",
    "particiones": 8,
    "destino": {"zona": "plata", "ruta": "/lake/plata"},
}
config_prod = {"particiones": 64, "compresion": "zstd"}

# Fusión: las claves de config_prod sobrescriben a las de config_base
config = {**config_base, **config_prod}
print(config)
print("Particiones:", config["particiones"])
print("Ruta destino:", config["destino"]["ruta"])

# Cuidado: el diccionario anidado "destino" es compartido
config["destino"]["zona"] = "oro"
print("Zona en config_base:", config_base["destino"]["zona"])

{'formato': 'parquet', 'compresion': 'zstd', 'particiones': 64, 'destino': {'zona': 'plata', 'ruta': '/lake/plata'}}
Particiones: 64
Ruta destino: /lake/plata
Zona en config_base: oro


**Paso a paso**

1. `{**a, **b}` crea un diccionario nuevo con todas las claves de `a` y de `b`. Si una clave está en los dos, gana el valor de `b`, que está más a la derecha.
2. La fusión es **superficial**: el diccionario anidado `destino` no se copia, sino que se comparte. Por eso, al cambiar `config["destino"]["zona"]` también cambia `config_base`. Es el mismo problema de alias que vimos en el Ejemplo 1.3.
3. Para copiar también los niveles internos se usa `copy.deepcopy()`.

#### Ejemplo 4.6 · Particionado por hash: qué partición le toca a cada clave

Spark y Kafka deciden en qué partición va cada registro calculando un número a partir de su clave (*hash*) y quedándose con el resto de dividirlo entre el número de particiones.

In [0]:
import zlib

num_particiones = 3
claves = ["C001", "C002", "C003", "C004", "C005", "C001", "C006", "C003"]

particiones = {}
for clave in claves:
    p = zlib.crc32(clave.encode()) % num_particiones
    particiones.setdefault(p, []).append(clave)

for p in sorted(particiones):
    print(f"Partición{p}:{particiones[p]}")

Partición0:['C003', 'C003']
Partición1:['C002', 'C005', 'C006']
Partición2:['C001', 'C004', 'C001']


**Paso a paso**

1. `zlib.crc32()` convierte el texto de la clave (pasado a bytes con `encode()`) en un número entero. Para la misma clave, el número es siempre el mismo.
2. `% num_particiones` convierte ese número en una partición entre 0 y 2.
3. `setdefault(p, [])` devuelve la lista de la partición y, si todavía no existe, la crea vacía. Así se puede hacer `append()` directamente.
4. Todos los registros de `C001` y de `C003` caen siempre en la misma partición. Por eso un `groupBy` o un `join` por cliente puede resolverse dentro de cada partición.
5. No se usa la función `hash()` de Python porque, para los textos, cambia en cada ejecución del programa. Un particionado debe ser estable entre ejecuciones y entre máquinas.

#### Ejemplo 4.7 · MapReduce con *combiner*: contar antes de enviar

En el Ejemplo 2.5 cada palabra viajaba por la red como un par `(palabra, 1)`. Un *combiner* hace un conteo **local** en cada nodo antes de enviar nada, y así reduce el volumen de datos del shuffle.

In [0]:
splits = [
    ["click", "view", "click", "view", "view", "compra"],
    ["view", "view", "click", "view"],
    ["click", "compra", "view", "click", "click"],
]

# MAP + COMBINER: cada nodo cuenta primero su propio split
parciales = []
for split in splits:
    local = {}
    for evento in split:
        local[evento] = local.get(evento, 0) + 1
    parciales.append(local)
    print("Conteo local:", local)

# REDUCE: se fusionan los conteos parciales
total = {}
for local in parciales:
    for evento, n in local.items():
        total[evento] = total.get(evento, 0) + n
print("Conteo global:", total)

sin_combiner = 0
con_combiner = 0
for i in range(len(splits)):
    sin_combiner += len(splits[i])
    con_combiner += len(parciales[i])
print("Pares enviados por la red sin combiner:", sin_combiner)
print("Pares enviados por la red con combiner:", con_combiner)

Conteo local: {'click': 2, 'view': 3, 'compra': 1}
Conteo local: {'view': 3, 'click': 1}
Conteo local: {'click': 3, 'compra': 1, 'view': 1}
Conteo global: {'click': 6, 'view': 7, 'compra': 2}
Pares enviados por la red sin combiner: 15
Pares enviados por la red con combiner: 8


**Paso a paso**

1. Cada nodo construye su propio diccionario de conteos con el patrón `get(clave, 0) + 1`.
2. En la fase reduce se recorren los diccionarios parciales y se suman sus valores en un diccionario global.
3. Sin combiner, cada evento viajaría por la red como un par: 15 pares. Con combiner, cada nodo envía un par por cada clave distinta: 8 pares.
4. Esta técnica funciona porque la suma es asociativa: da igual sumar por partes que todo junto. Las operaciones `reduceByKey` de Spark aplican esta optimización automáticamente.

### Ejercicios propuestos

#### Ejercicio 4.1 · Invertir un mapeo
Construye un diccionario region → lista de tiendas y muéstralo con las regiones y las tiendas ordenadas alfabéticamente. Pista: el método setdefault() permite crear la lista vacía la primera vez que aparece una región.

In [0]:
tienda_region = {
    "MAD01": "Centro", "MAD02": "Centro", "BCN01": "Noreste",
    "VLC01": "Levante", "BCN02": "Noreste", "TOL01": "Centro",
}
region_tiendas = {}
for tienda, region in tienda_region.items():
  region_tiendas.setdefault(region, []).append(tienda)

for region in sorted(region_tiendas.keys()):
  tiendas_ordenadas = sorted(region_tiendas[region])
  print(f"{region}->{tiendas_ordenadas  }")

Centro->['MAD01', 'MAD02', 'TOL01']
Levante->['VLC01']
Noreste->['BCN01', 'BCN02']


#### Ejercicio 4.2 · Total por tienda y mes con claves compuestas
Usa como clave una tupla `(tienda, mes)`, donde el mes son los 7 primeros caracteres de la fecha. Muestra los totales ordenados por tienda y mes, con dos decimales.

In [0]:
ventas = [
    ("2026-09-15", "MAD01", 200.0),
    ("2026-09-28", "BCN02", 150.0),
    ("2026-10-02", "MAD01", 320.0),
    ("2026-09-30", "MAD01", 80.0),
    ("2026-10-05", "BCN02", 410.0),
    ("2026-10-09", "MAD01", 95.5),
]

totales = {}
for fecha, tienda, importe in ventas:
    mes = fecha[:7]  
    clave = (tienda, mes)
    
    totales[clave] = totales.get(clave, 0.0) + importe

for (tienda, mes) in sorted(totales.keys()):
    print(f"{tienda} {mes}: {totales[(tienda, mes)]:.2f}")

BCN02 2026-09: 150.00
BCN02 2026-10: 410.00
MAD01 2026-09: 280.00
MAD01 2026-10: 415.50


#### Ejercicio 4.3 · Informe de nulos por campo
Un informe de calidad de datos necesita saber cuántos valores faltan en cada campo. Un valor cuenta como nulo si es `None`, si es una cadena vacía o si la clave no aparece en el registro.

In [0]:
registros = [
    {"id": 1, "email": "ana@acme.com", "ciudad": "Madrid", "edad": 34},
    {"id": 2, "email": None, "ciudad": "Sevilla", "edad": None},
    {"id": 3, "email": "luis@globex.com", "edad": 41},
    {"id": 4, "email": "", "ciudad": None, "edad": 29},
]
campos = ["id", "email", "ciudad", "edad"]

total_registros = len(registros)

for campo in campos:
    nulos = 0
    for reg in registros:
        if campo not in reg:
            nulos += 1
        else:
            valor = reg[campo]
            if valor is None or valor == "":
                nulos += 1
                
    porcentaje = (nulos / total_registros) * 100
    
    print(f"{campo:<7} {nulos} nulos ({porcentaje:.0f} %)")


id      0 nulos (0 %)
email   2 nulos (50 %)
ciudad  2 nulos (50 %)
edad    1 nulos (25 %)


#### Ejercicio 4.4 · Ranking de productos más vendidos
Suma las unidades por producto. Después ordena los pares de `items()` por unidades, de mayor a menor, con `itemgetter(1)`, y muestra el top 3 numerado.

In [0]:
from operator import itemgetter

lineas = [
    ("teclado", 3), ("raton", 10), ("monitor", 2), ("raton", 5),
    ("cable_hdmi", 8), ("teclado", 4), ("monitor", 1), ("cable_hdmi", 9),
]

totales = {}
for producto, unidades in lineas:
    totales[producto] = totales.get(producto, 0) + unidades

print(totales)


productos_ordenados = sorted(totales.items(), key=itemgetter(1), reverse=True)


for i, (producto, unidades) in enumerate(productos_ordenados[:3], start=1):
    print(f"{i}. {producto} ({unidades} uds)")


{'teclado': 7, 'raton': 15, 'monitor': 3, 'cable_hdmi': 17}
1. cable_hdmi (17 uds)
2. raton (15 uds)
3. teclado (7 uds)


#### Ejercicio 4.5 · Configuración en tres niveles

Un pipeline combina tres niveles de configuración. El nivel `ejecucion` tiene prioridad sobre `entorno`, y `entorno` sobre `defaults`.
Fusiona los tres niveles en un solo diccionario. Después indica, para cada clave, de qué nivel procede su valor final.

In [0]:
defaults = {"formato": "csv", "separador": ",", "reintentos": 3, "timeout_s": 60}
entorno = {"formato": "parquet", "timeout_s": 120}
ejecucion = {"reintentos": 5}


resultado = defaults | entorno | ejecucion

print(resultado)

for clave, valor in resultado.items():
    if clave in ejecucion:
        origen = "ejecucion"
    elif clave in entorno:
        origen = "entorno"
    else:
        origen = "defaults"
        
    print(f"{clave} = {valor} (viene de {origen})")


{'formato': 'parquet', 'separador': ',', 'reintentos': 5, 'timeout_s': 120}
formato = parquet (viene de entorno)
separador = , (viene de defaults)
reintentos = 5 (viene de ejecucion)
timeout_s = 120 (viene de entorno)


#### Ejercicio 4.6 · Lag de varios grupos de consumidores de Kafka

Varios equipos leen el mismo topic, cada uno con su propio grupo de consumidores. Hay que revisar cuál va retrasado.
Para cada grupo:

1. Construye un diccionario `partición → lag`. Si un grupo no tiene offset confirmado para una partición, considera que no ha leído nada (offset 0).
2. Calcula el lag total y la partición más retrasada. Pista: `max(lags, key=lags.get)` devuelve la clave con el valor más alto.
3. Marca el grupo como `ALERTA` si su lag total supera el umbral, o como `OK` en caso contrario.

In [0]:
ultimo_offset = {0: 15230, 1: 14980, 2: 15410, 3: 9800}
grupos = {
    "etl_lakehouse": {0: 15230, 1: 14975, 2: 15400, 3: 9800},
    "alertas_fraude": {0: 13900, 1: 14980, 2: 12010, 3: 9790},
    "dashboard_rt": {0: 15229, 1: 14980, 2: 15410},
}
umbral_alerta = 1000

for nombre_grupo, offsets_confirmados in grupos.items():
    lags = {}

    for particion, offset_maximo in ultimo_offset.items():
        offset_confirmado = offsets_confirmados.get(particion, 0)
        lags[particion] = offset_maximo - offset_confirmado

    lag_total = sum(lags.values())
    peor_particion = max(lags, key=lags.get)
    max_lag_particion = lags[peor_particion]

    estado = "ALERTA" if lag_total > umbral_alerta else "OK"

    print(
        f"{nombre_grupo} lag {lag_total} | "
        f"peor partición {peor_particion} ({max_lag_particion}) | "
        f"{estado}"
    )


etl_lakehouse lag 15 | peor partición 2 (10) | OK
alertas_fraude lag 4740 | peor partición 2 (3400) | ALERTA
dashboard_rt lag 9801 | peor partición 3 (9800) | ALERTA


#### Ejercicio 4.7 · Simular un *broadcast join*

Cuando una tabla es muy pequeña, Spark envía una copia completa a cada nodo (*broadcast*). Así cada partición de la tabla grande se cruza localmente, sin mover sus datos por la red.

1. Para cada partición, enriquece cada venta con su continente usando el diccionario `paises`. Los países que no estén en él se asignan a `"Desconocido"`.
2. Calcula en cada partición un diccionario local `continente → importe` y muéstralo.
3. Fusiona los resultados locales en un total global y muéstralo ordenado por continente, con dos decimales.

In [0]:
paises = {"ES": "Europa", "PT": "Europa", "MX": "América", "AR": "América", "JP": "Asia"}

particiones = [
    [("V1", "ES", 120.0), ("V2", "MX", 80.0), ("V3", "ES", 45.5)],
    [("V4", "JP", 300.0), ("V5", "AR", 60.0), ("V6", "FR", 99.0)],
    [("V7", "PT", 210.0), ("V8", "MX", 150.0)],
]

totales_locales = []
for i, particion in enumerate(particiones):
    local = {}
    for id_venta, pais, importe in particion:
        continente = paises.get(pais, "Desconocido")
        local[continente] = local.get(continente, 0.0) + importe
    totales_locales.append(local)
    print(f"Partición {i}: {local}")

total_global = {}
for local in totales_locales:
    for continente, importe in local.items():
        total_global[continente] = total_global.get(continente, 0.0) + importe

print("Ventas por continente:")
for continente in sorted(total_global):
    print(f"  {continente:<12} {total_global[continente]:6.2f}")

Partición 0: {'Europa': 165.5, 'América': 80.0}
Partición 1: {'Asia': 300.0, 'América': 60.0, 'Desconocido': 99.0}
Partición 2: {'Europa': 210.0, 'América': 150.0}
Ventas por continente:
  América      290.00
  Asia         300.00
  Desconocido   99.00
  Europa       375.50
